# AS02 — Finetuning Gemma-3-270m-it on Colab (A100)

Runs the same LLMBox experiments as on the laptop (`as02/run_experiments.sh`), but on a GPU.
Runtime -> Change runtime type -> **A100** (or L4/T4; then use `DTYPE=float32` for T4).

**Before running**: in Colab *Secrets* (key icon on the left) add
* `HF_TOKEN` — Hugging Face read token (Gemma licence must be accepted on the HF page)
* `GH_TOKEN` — GitHub token with `repo` scope (the repo is private) — or upload the repo zip instead.


In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# 1. get the code (private repo) -------------------------------------------
from google.colab import userdata
import os
GH = userdata.get('GH_TOKEN')
!git clone https://{GH}@github.com/yarayaocmu/95864-AS02-finetune-llmbox.git llmbox
%cd llmbox
!git log --oneline | head -3

In [ ]:
# 2. dependencies (Colab already has torch) ---------------------------------
!pip install -q -r requirements_as02.txt
import transformers, peft; print('transformers', transformers.__version__, 'peft', peft.__version__)

In [ ]:
# 3. models --------------------------------------------------------------------
from huggingface_hub import login, snapshot_download
login(token=userdata.get('HF_TOKEN'))
snapshot_download('google/gemma-3-270m-it', local_dir='models/llms/google/gemma-3-270m-it')
# judge model for Part D (different model family from the one under test)
snapshot_download('microsoft/Phi-4-mini-instruct', local_dir='models/llms/microsoft/phi4-mini-instruct')
os.environ['AS02_JUDGE_MODEL'] = os.path.abspath('models/llms/microsoft/phi4-mini-instruct')
!ls models/llms/google/gemma-3-270m-it

In [ ]:
# 4. Part A: LLMBox prepare_data (re-split with the same seed) + tokenizer/batch analysis
!HF_HUB_OFFLINE=1 python3 -m startllm mode=prepare_data data=jsonl \
    data.path=data/as02/transformed/medrx_experience.jsonl data.output_dir=data/as02/splits \
    data.train_fraction=0.8 model=gemma3_270m model.source=local \
    model.local_path=models/llms/google/gemma-3-270m-it training.max_length=512 2>&1 | grep '\[info\]'
!python3 as02/part_a_restore_meta.py
!python3 as02/part_a_tokenize_batches.py --model models/llms/google/gemma-3-270m-it | head -30

In [ ]:
# 5. Part C: experiments ------------------------------------------------------
%env PY=python3
%env DTYPE=bfloat16
!bash as02/run_experiments.sh exp1
!bash as02/run_experiments.sh exp2
!bash as02/run_experiments.sh exp3      # optional: attention-only LoRA
!python3 as02/part_c_compare_runs.py

In [ ]:
from IPython.display import Image, display
display(Image('data/as02/part_c/learning_curves.png'))

In [ ]:
# 6. Part D: 100-prompt evaluations (base vs finetuned), unseen drugs, edge cases
!python3 as02/part_d_generate_eval.py --model models/llms/google/gemma-3-270m-it --data data/as02/splits/test.jsonl --n 100 --tag base_test | tail -30
!python3 as02/part_d_generate_eval.py --model finetuned_exp1_baseline/merged --data data/as02/splits/test.jsonl --n 100 --tag exp1_test | tail -30
!python3 as02/part_d_generate_eval.py --model finetuned_exp2_updated/merged --data data/as02/splits/test.jsonl --n 100 --tag exp2_test | tail -30
!python3 as02/part_d_generate_eval.py --model finetuned_exp2_updated/merged --data data/as02/eval/unseen_drugs_eval.jsonl --n 100 --tag exp2_unseen | tail -30
!python3 as02/part_d_generate_eval.py --model finetuned_exp2_updated/merged --data data/as02/eval/edge_cases.jsonl --n 24 --max-new-tokens 40 --tag exp2_edge | tail -30

In [ ]:
# 7. Part D: LLM judge (Pydantic model, local Phi-4-mini as judge) over the finetuned model's answers
!python3 as02/part_d_llm_judge.py --evalset data/as02/part_d/evalset_exp2_test.jsonl | tail -20

In [ ]:
# 8. save results back to GitHub (adapters are small; merged models are ignored) -------
!git config user.email "yarayaocmu2027@gmail.com" && git config user.name "Yara Yao"
!git add data/evaluations data/as02 && git commit -q -m "AS02: Colab A100 results (exp1/exp2/exp3, Part D evals)" && git push -q origin HEAD
# and a zip you can download
!zip -qr as02_results.zip data/evaluations data/as02/part_a data/as02/part_c data/as02/part_d finetuned_exp1_baseline/adapter_model.safetensors finetuned_exp2_updated/adapter_model.safetensors
from google.colab import files; files.download('as02_results.zip')

## Human evaluation (interactive, must be done by each student)
Run in a Colab cell (it prompts in the output area) or locally after pulling the results:
```
!python3 -m startllm mode=evaluate data=jsonl data.path=data/as02/part_d/evalset_exp2_test.jsonl model.local_path=finetuned_exp2_updated/merged
```
When asked for an LLM-judge function paste: `from src.pydantic_models.experience_judge import judge_fn`
